In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/nova-tf-idf/cm_tfidf.png
/kaggle/input/datasets/raesiow/nova-tf-idf/baseline_preds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_text_vs_both.png
/kaggle/input/datasets/raesiow/gi-dataset/gi_table_folds.csv
/kaggle/input/datasets/raesiow/gi-dataset/atkinson_merged.csv
/kaggle/input/datasets/raesiow/gi-dataset/atkinson_off_fuzzy_matches.csv
/kaggle/input/datasets/raesiow/nova-distilbert-oof/__huggingface_repos__.json
/kaggle/input/datasets/raesiow/nova-distilbert-oof/distilbert_oof.csv


In [3]:
nova = pd.read_csv("/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv", dtype={"off_code": str})  # 1,884 NOVA training rows, barcode kept as text
gi_table = pd.read_csv("/kaggle/input/datasets/raesiow/gi-dataset/gi_table_folds.csv", dtype={"code": str})  # 6,851 GI products, barcode kept as text

print(nova.shape)  # expect (1884, 22)
print(gi_table.shape)  # expect (6851, 16)
print(nova.columns.tolist())  # column names, so we know which one holds the ingredient text

(1884, 22)
(6851, 16)
['fpx_code', 'fpx_name', 'nova', 'off_code', 'off_name', 'source', 'score', 'cos', 'protein_g_f', 'fat_g_f', 'carb_excl_fiber_g_f', 'sugars_g_f', 'fiber_g_f', 'sat_fat_g_f', 'sodium_g_f', 'ingredients_text', 'categories_en', 'pnns_groups_2', 'image_url', 'nova_group', 'group', 'fold']


In [4]:
nova_codes = set(nova["off_code"])  # all barcodes in the NOVA training rows
gi_in_nova_code = gi_table["code"].isin(nova_codes)  # True for GI products whose barcode is also in NOVA training

nova_texts = set(nova["ingredients_text"].str.lower().str.strip())  # NOVA ingredient texts, lowercased, spaces trimmed at the ends
gi_texts = gi_table["ingredients_text"].str.lower().str.strip()  # same cleaning for the GI products
gi_in_nova_text = gi_texts.isin(nova_texts)  # True for GI products whose text is identical to a NOVA training text

print("same barcode:", gi_in_nova_code.sum(), "products,", round(gi_in_nova_code.mean() * 100, 1), "%")  # count and percent
print("same text:", gi_in_nova_text.sum(), "products,", round(gi_in_nova_text.mean() * 100, 1), "%")  # count and percent
print("either:", (gi_in_nova_code | gi_in_nova_text).sum(), "products")  # | means "or": overlapping by barcode or text
print(gi_table["code"].head(3).tolist(), nova["off_code"].head(3).tolist())  # quick look that both barcodes look the same (text, leading zeros kept)

same barcode: 238 products, 3.5 %
same text: 466 products, 6.8 %
either: 466 products
['853923002244', '4008471505006', '4027394003161'] ['16000124950', '61483054577', '815154020251']


In [5]:
gi_table["in_nova_training"] = gi_in_nova_text  # new column: True for the 466 products DistilBERT will have seen

oof_path = '/kaggle/input/datasets/raesiow/nova-distilbert-oof/distilbert_oof.csv'
oof = pd.read_csv(oof_path)  # load it (uses the path found above)
print(oof.shape)  # expect 1884 rows
print(oof.columns.tolist())  # column names: I need to see if it has an id or text column, or just p1-p4
print(oof.head(3))  # first 3 rows

(1884, 8)
['fpx_code', 'off_code', 'nova', 'fold', 'p1', 'p2', 'p3', 'p4']
   fpx_code      off_code  nova  fold        p1        p2        p3        p4
0  57241000   16000124950     4     2  0.008805  0.002926  0.004129  0.984140
1  82101500   61483054577     2     0  0.971725  0.009705  0.011705  0.006865
2  92651000  815154020251     4     2  0.054991  0.004652  0.005985  0.934372


In [6]:
oof = pd.read_csv(oof_path, dtype={"off_code": str})  # reload the OOF file, barcode kept as text

same_order = (oof["off_code"].values == nova["off_code"].values).all()  # True only if every row has the same barcode in both tables
print("same row order:", same_order)  # must be True before we continue

oof["text_key"] = nova["ingredients_text"].str.lower().str.strip()  # copy the cleaned text from nova (safe because row order matches)

prob_cols = ["p1", "p2", "p3", "p4"]  # the four NOVA probability columns
oof_by_text = oof.groupby("text_key")[prob_cols].mean().reset_index()  # one row per unique text, probabilities averaged if repeated
print(oof_by_text.shape)  # number of unique texts (a bit under 1884)

overlap_texts = set(gi_texts[gi_table["in_nova_training"]])  # the texts of the 466 overlapping GI products
print("overlap texts found in OOF table:", len(overlap_texts & set(oof_by_text["text_key"])), "of", len(overlap_texts))  # & keeps texts in both: the two numbers should be equal

same row order: True
(1821, 5)
overlap texts found in OOF table: 253 of 253


In [7]:
print(gi_table.loc[gi_table["in_nova_training"], ["product_name", "ingredients_text"]].sample(8, random_state=1))  # 8 random overlapping GI products

                            product_name  \
3404  100% Pure Canadian Red Maple Syrup   
6159               Sweetened Apple Sauce   
3337                        kidney beans   
2275                         Blueberries   
707                               Apples   
2392                    mandarin oranges   
545                          Rolled Oats   
2533                 2% Reduced Fat Milk   

                                       ingredients_text  
3404                                       Maple Syrup.  
6159  Apples, Sugars (glucose-fructose), Water, Asco...  
3337  Organic kidney beans, water, organic lemon jui...  
2275                                          blueberry  
707                                               apple  
2392  Mandarin oranges, white grape juice from conce...  
545                             whole grain rolled oats  
2533  Organic grade a reduced fat milk, vitamin a pa...  


In [8]:
import torch                                  # PyTorch, the library DistilBERT runs on

print(torch.cuda.is_available())              # expect True (GPU is on); if False, check Settings -> Accelerator
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")   # e.g. 'Tesla T4'


True
Tesla T4


In [9]:
df= nova

In [10]:
from transformers import AutoTokenizer   # Hugging Face tool that loads the matching tokenizer for a model

model_name = "distilbert-base-uncased"                   # "uncased" = lowercases text first, like our TF-IDF did
tok = AutoTokenizer.from_pretrained(model_name)          # download the tokenizer (needs Internet ON)



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
from torch.utils.data import Dataset    # the base class PyTorch expects

df["label"] = df["nova"] - 1             # PyTorch wants classes numbered 0,1,2,3 (not 1,2,3,4); NOVA 1 -> 0, ..., NOVA 4 -> 3
MAX_LEN = 256                            # our cut-off from the length check

class NovaDataset(Dataset):                                   # our recipe, built on PyTorch's Dataset
    def __init__(self, texts, labels):                        # runs once when we create it
        self.enc = tok(list(texts),                           # tokenise every text at once
                       truncation=True,                       # cut texts longer than MAX_LEN
                       max_length=MAX_LEN,
                      return_token_type_ids = False)
        self.labels = list(labels)                            # store the labels as a plain list

    def __len__(self):                                        # "how many items?"
        return len(self.labels)

    def __getitem__(self, i):                                 # "give me item i"
        item = {key: torch.tensor(val[i])                     # token IDs + attention mask for row i, as tensors
                for key, val in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])         # the answer for row i
        return item


In [12]:
from torch.utils.data import DataLoader                                  # serves the Dataset in batches
from transformers import DataCollatorWithPadding, AutoModelForSequenceClassification

device = torch.device("cuda")                       # the GPU
collate = DataCollatorWithPadding(tok)              # pads each batch to its own longest text (with 0s in attention_mask)



In [13]:
from torch.optim import AdamW  # the optimiser: decides how big each weight nudge is
from transformers import get_linear_schedule_with_warmup  # learning-rate schedule: ramp up, then slowly down
from sklearn.utils.class_weight import compute_class_weight  # computes "balanced" class weights
import numpy as np  # numbers and arrays
import time  # to time the training

EPOCHS = 10  # passes over the training data (same as the 0.700 run)
BATCH = 16  # texts per training step
LR = 2e-5  # learning rate: small, so we don't wreck the pretrained knowledge

torch.manual_seed(42)  # fix randomness (new layer's start, shuffling) so a rerun gives the same model

train_loader = DataLoader(NovaDataset(df["ingredients_text"], df["label"]),  # ALL 1,884 rows, no folds
                          batch_size=BATCH, shuffle=True,  # new random order each epoch
                          collate_fn=collate)  # pad each batch

model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=4).to(device)  # fresh pretrained DistilBERT + new 4-class layer, on the GPU

w = compute_class_weight("balanced", classes=np.arange(4), y=df["label"])  # rarer class -> bigger weight, from all rows
print("class weights:", w.round(2))  # NOVA 2 should get by far the biggest weight
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float).to(device))  # weighted loss on the GPU

opt = AdamW(model.parameters(), lr=LR)  # optimiser over all of DistilBERT's weights
steps = EPOCHS * len(train_loader)  # total training steps
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)  # first 10%: LR rises from 0; then falls to 0

start = time.time()  # start the clock
for epoch in range(EPOCHS):  # one pass over all 1,884 rows per epoch
    model.train()  # training mode (turns on dropout)
    total_loss = 0  # add up the loss to print an average
    for batch in train_loader:  # 16 texts at a time
        batch = {k: v.to(device) for k, v in batch.items()}  # move the batch to the GPU
        labels = batch.pop("labels")  # take the answers out, so the model doesn't compute its own (unweighted) loss
        out = model(**batch)  # run the model: gives logits, 4 scores per text
        loss = loss_fn(out.logits, labels)  # weighted loss: how wrong were we?
        loss.backward()  # work out which way to nudge each weight
        opt.step()  # nudge the weights
        sched.step()  # update the learning rate for the next step
        opt.zero_grad()  # clear the nudges before the next batch
        total_loss += loss.item()  # keep a running total
    print(f"epoch {epoch + 1}: average loss {total_loss / len(train_loader):.3f}, "
          f"{time.time() - start:.0f} s so far")  # loss should go down epoch by epoch

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


class weights: [0.99 7.72 1.78 0.44]
epoch 1: average loss 1.254, 32 s so far
epoch 2: average loss 0.867, 64 s so far
epoch 3: average loss 0.565, 98 s so far
epoch 4: average loss 0.428, 134 s so far
epoch 5: average loss 0.299, 172 s so far
epoch 6: average loss 0.216, 208 s so far
epoch 7: average loss 0.180, 245 s so far
epoch 8: average loss 0.152, 282 s so far
epoch 9: average loss 0.131, 318 s so far
epoch 10: average loss 0.114, 355 s so far


In [16]:
check = NovaDataset(df["ingredients_text"][:200], df["label"][:200])  # first 200 NOVA training rows
check_loader = DataLoader(check, batch_size=64, shuffle=False, collate_fn=collate)  # keep order

model.eval()  # prediction mode
preds = []  # predicted classes go here
with torch.no_grad():  # no gradients needed
    for batch in check_loader:  # 64 rows at a time
        batch = {key: v.to(device) for key, v in batch.items()}  # move to GPU
        batch.pop("labels")  # model doesn't need the answers
        preds.append(model(**batch).logits.argmax(dim=1).cpu())  # class with the highest score

preds = torch.cat(preds).numpy()  # one predicted class per row
print("training accuracy:", (preds == df["label"][:200].values).mean())  # trained: expect above 0.9

training accuracy: 0.985


In [17]:
gi_dataset = NovaDataset(gi_table["ingredients_text"], [0] * len(gi_table))  # GI texts + pretend labels (all 0), never used
gi_loader = DataLoader(gi_dataset, batch_size=64, shuffle=False, collate_fn=collate)  # keep row order; bigger batches fine when not training

model.eval()  # prediction mode (turns off dropout)
probs = []  # one small array per batch goes in here
with torch.no_grad():  # no gradients needed: faster, less memory
    for batch in gi_loader:  # 64 GI products at a time
        batch = {key: v.to(device) for key, v in batch.items()}  # move batch to GPU
        batch.pop("labels")  # remove the pretend labels
        logits = model(**batch).logits  # 4 class scores per product
        probs.append(torch.softmax(logits, dim=1).cpu())  # scores -> probabilities that add to 1, back to CPU
gi_probs = torch.cat(probs).numpy()  # stick batches together -> (6851, 4) array

print(gi_probs.shape)  # expect (6851, 4)
for i, col in enumerate(prob_cols):  # prob_cols = ["p1", "p2", "p3", "p4"] from the lookup cell
    gi_table[col] = gi_probs[:, i]  # column i of the array becomes p1, p2, p3 or p4
print(gi_table[prob_cols].sum(axis=1).describe())  # every row should add up to 1
print(gi_table[prob_cols].idxmax(axis=1).value_counts())  # how many products land in each predicted NOVA group

(6851, 4)
count    6.851000e+03
mean     1.000000e+00
std      5.620092e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
p4    3648
p1    2036
p3     785
p2     382
Name: count, dtype: int64


In [18]:
gi_table["text_key"] = gi_texts  # cleaned text (lowercase, trimmed), made in the overlap cell
lookup = oof_by_text.set_index("text_key")  # look up probabilities by text
rows = gi_table["in_nova_training"]  # True for the 466 products the final model has seen

before = gi_table.loc[rows, prob_cols].copy()  # keep the old values, just to compare
gi_table.loc[rows, prob_cols] = lookup.loc[gi_table.loc[rows, "text_key"], prob_cols].values  # replace with out-of-fold probs
print("rows swapped:", rows.sum())  # expect 466
print("mean top probability before:", before.max(axis=1).mean().round(3))  # seen-in-training: expect very confident
print("mean top probability after: ", gi_table.loc[rows, prob_cols].max(axis=1).mean().round(3))  # out-of-fold: expect a bit lower

out = gi_table[["code", "in_nova_training"] + prob_cols]  # barcode, flag and the 4 probabilities, in gi_table row order
out.to_csv("/kaggle/working/gi_nova_probs.csv", index=False)  # save
print(out.shape)  # expect (6851, 6)

rows swapped: 466
mean top probability before: 0.948
mean top probability after:  0.909
(6851, 6)


/tmp/ipykernel_97/112519041.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.01023605 0.95957088 0.89682281 0.96604007 0.96604007 0.96604007
 0.96604007 0.96604007 0.96604007 0.96604007 0.96604007 0.96604007
 0.96604007 0.96604007 0.96604007 0.96604007 0.96604007 0.0063996
 0.00720467 0.00854735 0.00848568 0.25574046 0.96613139 0.94630718
 0.98449373 0.98449373 0.98449373 0.98449373 0.98449373 0.98449373
 0.98449373 0.98449373 0.74477142 0.97633708 0.97539139 0.97813696
 0.94441503 0.91559881 0.96381444 0.96381444 0.96381444 0.84506202
 0.91559881 0.91559881 0.96381444 0.53207409 0.60219371 0.60219371
 0.60219371 0.60219371 0.97479838 0.65834254 0.98107129 0.98107129
 0.98107129 0.98107129 0.98107129 0.96613139 0.97539139 0.96141243
 0.57928157 0.96141243 0.95809656 0.01010765 0.97629559 0.91701525
 0.85609806 0.94290596 0.98613387 0.07257278 0.46504244 0.46504244
 0.98168361 0.0049018  0.97453737 